<a href="https://colab.research.google.com/github/yudithvega-art/bhm-mrsa-indonesia/blob/main/Refit_rev_04_BHM_MRSA_31082026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Refit — three-level zero-inflated, left-censored BHM (presumptive MRSA)
MRAN-D-26-00126. **All farms kept** (incl. non-detecting farms).

Reports: grand mean (CFU/unit); variance components tau/omega/sigma with 95% CrI; zero-inflation posterior per matrix; LOO-CV model comparison (zero-inflated vs censoring-only); occurrence / conditional / marginal exposure (mean + tail-robust simulation); prior predictive check.

**Colab:** run install cell -> upload `MRSA_replicate_level_modelready.xlsx` -> Run all.

In [1]:
!pip -q install "pymc>=5" arviz openpyxl pandas numpy h5netcdf

In [2]:
import os
if not os.path.exists('MRSA_replicate_level_modelready.xlsx'):
    try:
        from google.colab import files
        files.upload()
    except Exception as e:
        print('Put the xlsx in the Files panel.', e)

Saving MRSA_replicate_level_modelready.xlsx to MRSA_replicate_level_modelready.xlsx


In [4]:
# ============================================================================
#  Refit of the three-level, zero-inflated, left-censored Bayesian
#  Hierarchical Model (BHM) for presumptive MRSA concentrations.
#  Matches Eqs. (1)-(4) and the current priors (MRAN-D-26-00126).
#
#  ALL farms are kept (incl. farms with no detections): their 0-colony plates
#  enter via the zero-inflation + left-censoring components, so estimates are
#  NOT conditional on detection.
#
#  COLAB:  !pip -q install "pymc>=5" arviz openpyxl pandas numpy h5netcdf
#          upload MRSA_replicate_level_modelready.xlsx  ->  Run all.
# ============================================================================
import os
import numpy as np, pandas as pd, pymc as pm, arviz as az, pytensor.tensor as pt

DATA_PATH = "MRSA_replicate_level_modelready.xlsx"
# try:
#     from google.colab import files
#     if not os.path.exists(DATA_PATH): DATA_PATH = list(files.upload().keys())[0]
# except Exception: pass
df = pd.read_excel(DATA_PATH, sheet_name="replicate_data").dropna(subset=["matrix"]).reset_index(drop=True)

ETA_PRIOR = {"Milk": (4.0, 1.2), "Water": (2.0, 1.0), "Bioaerosol": (2.0, 0.8), "Dust": (1.0, 0.8)}
UNIT = {"Milk": "CFU/mL", "Water": "CFU/100 mL", "Bioaerosol": "CFU/m3", "Dust": "CFU/cm2"}
DRAWS, TUNE, CHAINS, SEED, TARGET = 2000, 2500, 4, 42, 0.95

def _q(x):
    a, b, c = np.percentile(np.asarray(x), [2.5, 50, 97.5]); return float(b), float(a), float(c)

# ---- model builder / fit (zero_inflated True/False); CustomDist -> LOO/WAIC ----
def fit_matrix(m, zero_inflated=True, draws=DRAWS, tune=TUNE, chains=CHAINS, cores=None, seed=SEED):
    d = df[df.matrix == m].copy().reset_index(drop=True)
    farms = sorted(d.farm.unique()); fidx = {f: i for i, f in enumerate(farms)}
    d["pt_id"] = d.farm.astype(str) + "||" + d.point.astype(str)
    points = sorted(d.pt_id.unique()); pidx = {p: i for i, p in enumerate(points)}
    J, K = len(farms), len(points)
    point_farm = np.array([fidx[p.split("||")[0]] for p in points]); d["k"] = d.pt_id.map(pidx)
    LOD = float(d.LOD.iloc[0]); logLOD = np.log10(LOD)
    k_obs = d.k.values.astype("int64"); is_det = (d.detection.values == 1).astype("float64")
    conc = d.conc.values.astype("float64")
    value = np.where(is_det > 0.5, np.log10(np.where(conc > 0, conc, 1.0)), logLOD)
    mu_eta, sd_eta = ETA_PRIOR[m]

    def obs_logp(value, mu_, pi_, sig_, isdet_, logLOD_):
        nll = pm.logp(pm.Normal.dist(mu_, sig_), value)
        det_ll = pt.log1p(-pi_) + nll
        z_c = (logLOD_ - mu_) / sig_; Phi = 0.5 * pt.erfc(-z_c / pt.sqrt(2.0))
        cen_ll = pt.log(pi_ + (1.0 - pi_) * Phi + 1e-12)
        return pt.where(isdet_ > 0.5, det_ll, cen_ll)

    with pm.Model() as model:
        eta = pm.Normal("eta", mu_eta, sd_eta)
        tau = pm.HalfNormal("tau", 0.6); omega = pm.HalfNormal("omega", 0.6); sigma = pm.HalfNormal("sigma", 1.0)
        z_theta = pm.Normal("z_theta", 0, 1, shape=J); theta = pm.Deterministic("theta", eta + tau * z_theta)
        z_mu = pm.Normal("z_mu", 0, 1, shape=K); mu = pm.Deterministic("mu", theta[point_farm] + omega * z_mu)
        if zero_inflated:
            delta = pm.HalfNormal("delta", 1.0); alpha = pm.Normal("alpha", 0, 1.5, shape=J)
            z_logit = pm.Normal("z_logit", 0, 1, shape=K)
            pi = pm.Deterministic("pi", pm.math.sigmoid(alpha[point_farm] + delta * z_logit))
        else:
            pi = pm.Deterministic("pi", pt.zeros(K))
        pm.CustomDist("obs", mu[k_obs], pi[k_obs], sigma, pt.as_tensor_variable(is_det), logLOD,
                      logp=obs_logp, observed=value)
        idata = pm.sample(draws=draws, tune=tune, chains=chains, cores=cores,
                          target_accept=TARGET, random_seed=seed)
        pm.compute_log_likelihood(idata, model=model, progressbar=False)
    info = dict(J=J, K=K, LOD=LOD, n=len(d), n_det=int((is_det > 0.5).sum()),
                zero_farms=[f for f in farms if d[d.farm == f].detection.sum() == 0])
    return model, idata, info

def run_all(cores=None, draws=DRAWS, tune=TUNE, chains=CHAINS):
    rows, idatas = [], {}
    for m in ["Milk", "Water", "Bioaerosol", "Dust"]:
        print(f"\n=================  {m}  =================")
        _, idata, info = fit_matrix(m, True, draws, tune, chains, cores); idatas[m] = idata
        try: idata.to_netcdf(f"bhm_{m}.nc")
        except Exception as e: print(f"  (NetCDF not saved: {e})")
        post = idata.posterior; eta = post["eta"].values.reshape(-1)
        dvg = int(idata.sample_stats["diverging"].values.sum())
        s = az.summary(idata, var_names=["eta", "tau", "omega", "sigma", "delta"], round_to=3)
        rhat = float(s["r_hat"].max()); ess = float(s["ess_bulk"].min()); gm = _q(10 ** eta)
        print(s.to_string())
        print(f"  farms kept = {info['J']}/10 (0-detection farms still in model: {info['zero_farms'] or 'none'})")
        print(f"  grand-mean {UNIT[m]}: median {gm[0]:.4g}  95% CrI [{gm[1]:.3g}, {gm[2]:.4g}]")
        print(f"  diagnostics: max R-hat {rhat:.4f} | min ESS {ess:.0f} | divergences {dvg}")
        mc = lambda v: _q(post[v].values.reshape(-1))
        rows.append(dict(matrix=m, unit=UNIT[m], LOD=info["LOD"], K=info["K"], n=info["n"],
                         n_det=info["n_det"], zero_farms=len(info["zero_farms"]),
                         grand_mean=gm[0], gm_lo=gm[1], gm_hi=gm[2],
                         **{f"{v}_{q}": val for v in ["tau", "omega", "sigma", "delta"]
                            for q, val in zip(["med", "lo", "hi"], mc(v))},
                         max_rhat=rhat, min_ess=ess, divergences=dvg))
    summary = pd.DataFrame(rows); summary.to_csv("bhm_summary.csv", index=False)
    print("\n================  COMBINED SUMMARY  ================"); print(summary.to_string(index=False))
    return summary, idatas

# ---- variance components: median [95% CrI]  (Reviewer #2) ----
def report_variance(idatas):
    print("\n====  Variance components: posterior median [95% CrI] (log10 scale)  ====")
    print(f"{'Matrix':11}{'tau (between-farm)':27}{'omega (between-point)':27}{'sigma (residual)':22}")
    fmt = lambda x: f"{x[0]:.3f} [{x[1]:.3f}, {x[2]:.3f}]"
    for m, idata in idatas.items():
        p = idata.posterior
        print(f"{m:11}{fmt(_q(p['tau'].values.reshape(-1))):27}"
              f"{fmt(_q(p['omega'].values.reshape(-1))):27}{fmt(_q(p['sigma'].values.reshape(-1))):22}")

# ---- zero-inflation posterior per matrix (identifiability)  (Reviewer #5) ----
def report_zero_inflation(idatas):
    print("\n====  Zero-inflation posterior: structural-zero prob pi (median [95% CrI])  ====")
    for m, idata in idatas.items():
        p = idata.posterior
        pb = _q(p["pi"].values.mean(axis=-1).reshape(-1))
        d_ = _q(p["delta"].values.reshape(-1)) if "delta" in p else (float('nan'),)*3
        print(f"  {m:11} mean pi = {pb[0]:.2f} [{pb[1]:.2f}, {pb[2]:.2f}]   "
              f"delta = {d_[0]:.2f} [{d_[1]:.2f}, {d_[2]:.2f}]")
    print("  (posterior clearly tighter than the wide alpha~N(0,1.5) prior => identified)")

# ---- model comparison ZI vs censoring-only: LOO-CV  (Reviewer #5) ----
def compare_models(m, draws=DRAWS, tune=TUNE, chains=CHAINS, cores=None):
    print(f"\n====  Model comparison {m}: zero-inflated vs censoring-only (LOO-CV)  ====")
    _, zi, _ = fit_matrix(m, True,  draws, tune, chains, cores)
    _, ce, _ = fit_matrix(m, False, draws, tune, chains, cores)
    cmp = az.compare({"zero_inflated": zi, "censoring_only": ce})   # ELPD-LOO, ArviZ 0.x & 1.x
    print("(higher elpd = better; rank 0 = preferred model)")
    print(cmp.to_string())
    best  = cmp.sort_values("rank").index[0]
    other = [i for i in cmp.index if i != best][0]
    ediff = float(cmp.loc[other, "elpd_diff"]); dse = float(cmp.loc[other, "dse"])
    print(f"  -> preferred by LOO-CV: {best.replace('_',' ')}"
          f"  ({other.replace('_',' ')} worse by |elpd_diff| = {abs(ediff):.1f}, dse = {dse:.1f})")
    if abs(ediff) < 2 * dse:
        print("     (difference < 2*dse: models are close; report as weak/moderate support)")
    print("  If Pareto k > 0.7 is flagged, LOO is approximate for those influential points.")
    return cmp, zi, ce

# ---- occurrence / conditional / marginal exposure (+ tail-robust)  (Rev #5) ----
def decompose(idata, m, exposure_factor=1.0):
    p = idata.posterior; f = lambda v: p[v].stack(s=("chain", "draw")).values
    pi, mu, eta, sigma = f("pi"), f("mu"), f("eta"), f("sigma"); ln10 = np.log(10.0)
    marg = ((1 - pi) * (10.0 ** mu * np.exp(0.5 * ln10**2 * sigma**2))).mean(axis=0) * exposure_factor
    return dict(occurrence=_q((1 - pi).mean(axis=0)), conditional_median=_q(10.0 ** eta), marginal_mean=_q(marg))

def marginal_sim(idata, m, exposure_factor=1.0, n_pt=400, seed=SEED):
    p = idata.posterior; f = lambda v: p[v].stack(s=("chain", "draw")).values
    pi, mu, sigma = f("pi"), f("mu"), f("sigma"); K, S = pi.shape; rng = np.random.default_rng(seed)
    k = rng.integers(0, K, size=(S, n_pt))
    present = rng.random((S, n_pt)) < (1 - np.take_along_axis(pi.T, k, 1))
    logC = rng.normal(np.take_along_axis(mu.T, k, 1), sigma[:, None])
    C = np.where(present, 10.0 ** logC, 0.0) * exposure_factor
    return dict(marg_median=_q(np.median(C, 1)), marg_p95=_q(np.percentile(C, 95, 1)),
                marg_mean=_q(np.mean(C, 1)), zero_fraction=float(np.mean(C == 0)))

def report_exposure(idatas, exposure_factors=None):
    ef = exposure_factors or {k: 1.0 for k in idatas}
    print("\n====  Occurrence / conditional / marginal exposure (median [95% CrI])  ====")
    for m, idata in idatas.items():
        d = decompose(idata, m, ef.get(m, 1.0)); s = marginal_sim(idata, m, ef.get(m, 1.0))
        o, c = d["occurrence"], d["conditional_median"]
        print(f"\n{m} ({UNIT[m]}):   [{s['zero_fraction']*100:.0f}% of events = 0 (absence)]")
        print(f"  occurrence P(present)          = {o[0]:.2f} [{o[1]:.2f}, {o[2]:.2f}]")
        print(f"  conditional | present (median) = {c[0]:.4g} [{c[1]:.3g}, {c[2]:.4g}]")
        print(f"  marginal MEDIAN (tail-robust)  = {s['marg_median'][0]:.4g} [{s['marg_median'][1]:.3g}, {s['marg_median'][2]:.4g}]")
        print(f"  marginal 95th percentile       = {s['marg_p95'][0]:.4g} [{s['marg_p95'][1]:.3g}, {s['marg_p95'][2]:.4g}]")
        print(f"  marginal MEAN (expected dose)  = {s['marg_mean'][0]:.4g} [{s['marg_mean'][1]:.3g}, {s['marg_mean'][2]:.4g}]  (tail-driven)")

# ---- PER-FARM decomposition: occurrence / conditional / marginal per farm  (Reviewer #5) ----
def decompose_by_farm(idata, m, exposure_factor=1.0, n_sim=600, seed=SEED):
    """Returns a DataFrame with, for every farm j: occurrence P(present),
    conditional concentration | present (from theta_j), and marginal exposure
    (analytic mean + tail-robust simulated median), each as median [95% CrI]."""
    d = df[df.matrix == m].copy().reset_index(drop=True)
    farms = sorted(d.farm.unique()); fidx = {f: i for i, f in enumerate(farms)}
    d["pt_id"] = d.farm.astype(str) + "||" + d.point.astype(str)
    points = sorted(d.pt_id.unique())
    point_farm = np.array([fidx[p.split("||")[0]] for p in points])   # farm of each point

    p = idata.posterior
    f = lambda v: p[v].stack(s=("chain", "draw")).values
    theta = f("theta")                       # (J, S) farm mean (log10) -- conditional
    pi    = f("pi")                          # (K, S) structural-zero prob per point
    sigma = f("sigma")                       # (S,)
    ln10  = np.log(10.0)
    S = theta.shape[1]
    rng = np.random.default_rng(seed)

    rows = []
    for j, farm in enumerate(farms):
        kk = np.where(point_farm == j)[0]                 # points belonging to this farm
        occ_j = (1.0 - pi[kk]).mean(axis=0)               # farm occurrence P(present)  (S,)
        cond_j = 10.0 ** theta[j]                         # conditional median | present (S,)
        # marginal MEAN (analytic, lognormal mean at farm level)
        cond_mean_j = 10.0 ** theta[j] * np.exp(0.5 * ln10**2 * sigma**2)
        marg_mean_j = occ_j * cond_mean_j * exposure_factor
        # marginal MEDIAN (tail-robust) via simulation over this farm's points
        ki = rng.integers(0, len(kk), size=(S, n_sim))
        present = rng.random((S, n_sim)) < (1.0 - np.take_along_axis(pi[kk].T, ki, 1))
        # concentration when present ~ lognormal around theta_j (farm mean)
        logC = rng.normal(theta[j][:, None], sigma[:, None], size=(S, n_sim))
        C = np.where(present, 10.0 ** logC, 0.0) * exposure_factor
        marg_med_j = np.median(C, axis=1)
        oc = _q(occ_j); cn = _q(cond_j); mm = _q(marg_med_j); me = _q(marg_mean_j)
        rows.append(dict(farm=farm,
                         occ=oc[0], occ_lo=oc[1], occ_hi=oc[2],
                         cond=cn[0], cond_lo=cn[1], cond_hi=cn[2],
                         marg_median=mm[0], marg_median_lo=mm[1], marg_median_hi=mm[2],
                         marg_mean=me[0], marg_mean_lo=me[1], marg_mean_hi=me[2]))
    return pd.DataFrame(rows)


def report_by_farm(idatas, exposure_factors=None, save=True):
    ef = exposure_factors or {k: 1.0 for k in idatas}
    all_tbl = []
    for m, idata in idatas.items():
        tbl = decompose_by_farm(idata, m, ef.get(m, 1.0)); tbl.insert(0, "matrix", m)
        all_tbl.append(tbl)
        print(f"\n====  Per-farm occurrence / conditional / marginal  --  {m} ({UNIT[m]})  ====")
        print(f"{'farm':7}{'occurrence':22}{'conditional|present':24}{'marginal (median)':24}{'marginal (mean)'}")
        for _, r in tbl.iterrows():
            print(f"{r['farm']:7}"
                  f"{f'{r.occ:.2f} [{r.occ_lo:.2f},{r.occ_hi:.2f}]':22}"
                  f"{f'{r.cond:.4g} [{r.cond_lo:.3g},{r.cond_hi:.4g}]':24}"
                  f"{f'{r.marg_median:.4g} [{r.marg_median_lo:.3g},{r.marg_median_hi:.4g}]':24}"
                  f"{f'{r.marg_mean:.4g} [{r.marg_mean_lo:.3g},{r.marg_mean_hi:.4g}]'}")
    out = pd.concat(all_tbl, ignore_index=True)
    if save:
        out.to_csv("bhm_by_farm.csv", index=False)
        print("\nSaved per-farm table: bhm_by_farm.csv")
    return out


# ---- 6. PRIOR SENSITIVITY: shift the eta prior median +/-1 log10 and refit  (Reviewer #9) ----
def prior_sensitivity(matrices=None, shifts=(-1.0, 0.0, +1.0),
                      draws=DRAWS, tune=TUNE, chains=CHAINS, cores=None):
    """Refit each matrix with the grand-mean prior median shifted by `shifts`
    (in log10 units, i.e. x0.1 / x1 / x10) and report how much the POSTERIOR
    grand mean, occurrence and marginal exposure move. Small movement of the
    posterior relative to the 10-fold prior shift => data-dominated / robust."""
    matrices = matrices or ["Milk", "Water", "Bioaerosol", "Dust"]
    base = dict(ETA_PRIOR)          # keep originals to restore afterwards
    rows = []
    print("\n====  Prior sensitivity: eta prior median shifted +/-1 log10 (x0.1 / x1 / x10)  ====")
    for m in matrices:
        mu0, sd0 = base[m]
        print(f"\n-- {m} ({UNIT[m]}) : default eta ~ Normal({mu0}, {sd0}) --")
        print(f"{'shift':10}{'prior median':16}{'posterior grand mean':30}"
              f"{'occurrence':18}{'marginal mean':22}")
        for sh in shifts:
            ETA_PRIOR[m] = (mu0 + sh, sd0)          # temporarily shift the prior
            _, idata, _ = fit_matrix(m, True, draws, tune, chains, cores)
            eta = idata.posterior["eta"].values.reshape(-1)
            gm = _q(10 ** eta)
            dec = decompose(idata, m)
            occ, marg = dec["occurrence"], dec["marginal_mean"]
            tag = {-1.0: "-1 (x0.1)", 0.0: " 0 (base)", 1.0: "+1 (x10)"}.get(sh, f"{sh:+.1f}")
            print(f"{tag:10}{10**(mu0+sh):<16.4g}"
                  f"{f'{gm[0]:.4g} [{gm[1]:.3g},{gm[2]:.4g}]':30}"
                  f"{f'{occ[0]:.2f} [{occ[1]:.2f},{occ[2]:.2f}]':18}"
                  f"{f'{marg[0]:.4g} [{marg[1]:.3g},{marg[2]:.4g}]':22}")
            rows.append(dict(matrix=m, unit=UNIT[m], prior_shift_log10=sh,
                             prior_median=10 ** (mu0 + sh),
                             post_grand_mean=gm[0], gm_lo=gm[1], gm_hi=gm[2],
                             occurrence=occ[0], marginal_mean=marg[0]))
        ETA_PRIOR[m] = (mu0, sd0)                    # restore default prior
    tbl = pd.DataFrame(rows)
    # robustness ratio: posterior grand-mean fold-change vs the 100x prior swing
    for m in matrices:
        sub = tbl[tbl.matrix == m].set_index("prior_shift_log10")
        if -1.0 in sub.index and 1.0 in sub.index:
            fold = sub.loc[1.0, "post_grand_mean"] / max(sub.loc[-1.0, "post_grand_mean"], 1e-9)
            print(f"  {m:11}: posterior grand mean changes {fold:.2g}x across a 100x prior swing "
                  f"({'robust, data-dominated' if fold < 10 else 'prior-sensitive -- report caution'})")
    tbl.to_csv("bhm_prior_sensitivity.csv", index=False)
    print("\nSaved: bhm_prior_sensitivity.csv")
    return tbl


def prior_predictive_check():
    print("\n----  Prior predictive check  ----")
    for m in ["Milk", "Water", "Bioaerosol", "Dust"]:
        mu_e, sd_e = ETA_PRIOR[m]; rng = np.random.default_rng(SEED)
        eta = rng.normal(mu_e, sd_e, 200000)
        C = 10 ** rng.normal(rng.normal(rng.normal(eta, np.abs(rng.normal(0, .6, 200000))),
                                        np.abs(rng.normal(0, .6, 200000))), np.abs(rng.normal(0, 1., 200000)))
        print(f"  {m:11} median {np.median(C):.4g} {UNIT[m]:10} 95% [{np.percentile(C,2.5):.3g}, {np.percentile(C,97.5):.4g}]")

# ===========================================================================
#  VISUALISATION  (matplotlib). Each function saves a PNG and returns the fig.
#  Call plot_all(idatas, sens_tbl=..., byfarm_tbl=...) after run_all(), or run
#  the individual plot_* functions. Colab: figures also display inline.
# ===========================================================================
import matplotlib
matplotlib.use("Agg")            # remove this line on Colab to display inline
import matplotlib.pyplot as plt

_MC = {"Milk": "#4A72A6", "Water": "#2F8F8B", "Bioaerosol": "#E07A5F", "Dust": "#7B5EA7"}
_ORD = ["Milk", "Water", "Bioaerosol", "Dust"]


def _flat(idata, v):
    return idata.posterior[v].stack(s=("chain", "draw")).values


# ---- (1) grand-mean posterior + observed detections, per matrix -------------
def plot_grandmean(idatas, fname="fig1_grandmean.png"):
    fig, axs = plt.subplots(1, 4, figsize=(15, 3.6))
    for ax, m in zip(axs, [x for x in _ORD if x in idatas]):
        eta = _flat(idatas[m], "eta")                    # log10 grand mean
        C = 10.0 ** eta
        ax.hist(np.log10(C), bins=60, color=_MC[m], alpha=0.55, density=True)
        for p in np.percentile(np.log10(C), [2.5, 50, 97.5]):
            ax.axvline(p, color=_MC[m], ls="--" if p != np.median(np.log10(C)) else "-", lw=1.3)
        d = df[df.matrix == m]; det = d[d.detection == 1].conc.astype(float)
        if len(det):
            ax.plot(np.log10(det), np.full(len(det), -0.02), "|", color="k", ms=8, alpha=0.5)
        gm = 10 ** np.median(eta)
        ax.set_title(f"{m}\n grand mean {gm:.3g} {UNIT[m]}", fontsize=10, color=_MC[m], fontweight="bold")
        ax.set_xlabel("log10 concentration"); ax.set_yticks([])
    fig.suptitle("(1) Posterior grand mean per matrix  (ticks = observed detections; all 10 farms in model)",
                 fontsize=12, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.92]); fig.savefig(fname, dpi=170); return fig


# ---- (2) variance components tau / omega / sigma with 95% CrI ---------------
def plot_variance(idatas, fname="fig2_variance.png"):
    comps = ["tau", "omega", "sigma"]; labels = ["τ (between-farm)", "ω (between-point)", "σ (residual)"]
    mats = [x for x in _ORD if x in idatas]
    fig, ax = plt.subplots(figsize=(9, 4.4))
    x = np.arange(len(mats)); w = 0.25
    for i, (c, lab) in enumerate(zip(comps, labels)):
        med, lo, hi = [], [], []
        for m in mats:
            q = _q(_flat(idatas[m], c)); med.append(q[0]); lo.append(q[0]-q[1]); hi.append(q[2]-q[0])
        ax.bar(x + (i-1)*w, med, w, yerr=[lo, hi], capsize=3,
               color=["#4A72A6", "#2F8F8B", "#E07A5F"][i], label=lab, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(mats); ax.set_ylabel("SD on log10 scale")
    ax.set_title("(2) Variance components — posterior median ± 95% CrI", fontweight="bold")
    ax.legend(fontsize=9); fig.tight_layout(); fig.savefig(fname, dpi=170); return fig


# ---- (3a) zero-inflation posterior pi per matrix ----------------------------
def plot_zero_inflation(idatas, fname="fig3_zeroinflation.png"):
    mats = [x for x in _ORD if x in idatas]
    fig, ax = plt.subplots(figsize=(8, 4))
    prior = np.random.default_rng(0)
    a = prior.normal(0, 1.5, 200000); pi_prior = 1/(1+np.exp(-a))         # sigmoid(N(0,1.5))
    parts = ax.violinplot([_flat(idatas[m], "pi").mean(axis=0) for m in mats],
                          showmedians=True, showextrema=False)
    for b, m in zip(parts["bodies"], mats):
        b.set_facecolor(_MC[m]); b.set_alpha(0.6)
    ax.axhspan(np.percentile(pi_prior, 2.5), np.percentile(pi_prior, 97.5),
               color="grey", alpha=0.15, label="prior 95% (sigmoid α~N(0,1.5))")
    ax.set_xticks(range(1, len(mats)+1)); ax.set_xticklabels(mats)
    ax.set_ylabel("structural-zero prob  π"); ax.set_ylim(0, 1)
    ax.set_title("(3a) Zero-inflation posterior per matrix (tighter than prior ⇒ identified)", fontweight="bold")
    ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(fname, dpi=170); return fig


# ---- (3b) LOO-CV model comparison (needs compare_models results) ------------
def plot_loo_compare(compare_results, fname="fig3b_loo.png"):
    """compare_results: {matrix: az.compare DataFrame}."""
    mats = list(compare_results); fig, ax = plt.subplots(figsize=(8, 4))
    for i, m in enumerate(mats):
        cmp = compare_results[m]
        for model in cmp.index:
            elpd = cmp.loc[model, "elpd" if "elpd" in cmp.columns else "elpd_loo"]
            se = cmp.loc[model, "se"]
            col = "#2F8F8B" if model == "zero_inflated" else "#E07A5F"
            off = -0.15 if model == "zero_inflated" else 0.15
            ax.errorbar(i+off, elpd, yerr=se, fmt="o", color=col, capsize=4,
                        label=model.replace("_", " ") if i == 0 else None)
    ax.set_xticks(range(len(mats))); ax.set_xticklabels(mats)
    ax.set_ylabel("ELPD-LOO (higher = better)")
    ax.set_title("(3b) Model comparison: zero-inflated vs censoring-only (LOO-CV)", fontweight="bold")
    ax.legend(fontsize=9); fig.tight_layout(); fig.savefig(fname, dpi=170); return fig


# ---- (4a) occurrence / conditional / marginal CONCENTRATION at matrix level -
def plot_exposure(idatas, exposure_factors=None, fname="fig4_marginal_concentration.png"):
    ef = exposure_factors or {k: 1.0 for k in idatas}
    mats = [x for x in _ORD if x in idatas]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2))
    # occurrence
    for i, m in enumerate(mats):
        o = decompose(idatas[m], m, ef.get(m, 1.0))["occurrence"]
        a1.bar(i, o[0], yerr=[[o[0]-o[1]], [o[2]-o[0]]], capsize=4, color=_MC[m], alpha=0.85)
    a1.set_xticks(range(len(mats))); a1.set_xticklabels(mats); a1.set_ylim(0, 1)
    a1.set_ylabel("P(present)"); a1.set_title("occurrence probability", fontsize=10)
    # conditional median vs marginal median vs marginal mean (log scale)
    xw = 0.25
    for i, m in enumerate(mats):
        d = decompose(idatas[m], m, ef.get(m, 1.0)); s = marginal_sim(idatas[m], m, ef.get(m, 1.0))
        a2.bar(i-xw, d["conditional_median"][0], xw, color=_MC[m], alpha=0.9)
        a2.bar(i,    s["marg_median"][0],       xw, color=_MC[m], alpha=0.55)
        a2.bar(i+xw, s["marg_mean"][0],         xw, color=_MC[m], alpha=0.30)
    a2.set_yscale("log"); a2.set_xticks(range(len(mats))); a2.set_xticklabels(mats)
    a2.set_ylabel("presumptive-MRSA concentration (matrix unit)")
    a2.set_title("conditional | present  vs  marginal median  vs  marginal mean", fontsize=10)
    from matplotlib.patches import Patch
    a2.legend(handles=[Patch(fc="grey", alpha=.9, label="conditional | present"),
                       Patch(fc="grey", alpha=.55, label="marginal median (tail-robust)"),
                       Patch(fc="grey", alpha=.30, label="marginal mean")], fontsize=8)
    fig.suptitle("(4a) Occurrence & marginal CONCENTRATION (presumptive MRSA) — matrix level\n"
                 "environmental concentration, NOT exposure dose (no pathway factors applied)",
                 fontweight="bold", fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.90]); fig.savefig(fname, dpi=170); return fig


# ---- (4b) per-farm occurrence & marginal ------------------------------------
def plot_by_farm(byfarm_tbl, fname="fig4b_byfarm.png"):
    mats = [m for m in _ORD if m in byfarm_tbl.matrix.unique()]
    fig, axs = plt.subplots(1, len(mats), figsize=(4*len(mats), 4), sharey=False)
    if len(mats) == 1: axs = [axs]
    for ax, m in zip(axs, mats):
        t = byfarm_tbl[byfarm_tbl.matrix == m].sort_values("farm")
        y = np.arange(len(t))
        ax.barh(y, t["occ"], color=_MC[m], alpha=0.35, label="occurrence")
        ax2 = ax.twiny()
        ax2.plot(t["marg_median"]+1e-6, y, "o-", color=_MC[m], label="marginal median")
        ax2.set_xscale("symlog")
        ax.set_yticks(y); ax.set_yticklabels(t["farm"], fontsize=8)
        ax.set_xlim(0, 1); ax.set_title(f"{m}", color=_MC[m], fontweight="bold")
        ax.set_xlabel("occurrence"); ax2.set_xlabel(f"marginal median ({UNIT[m]})", fontsize=8)
    fig.suptitle("(4b) Per-farm occurrence (bars) & marginal-median exposure (points)", fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.90]); fig.savefig(fname, dpi=170); return fig


# ---- (5) prior predictive check ---------------------------------------------
def plot_prior_predictive(fname="fig5_priorpred.png"):
    lit = {"Milk": (300, 200000), "Water": None, "Bioaerosol": (1, 2000), "Dust": (2, 630)}
    fig, axs = plt.subplots(2, 2, figsize=(11, 7)); axs = axs.ravel()
    for ax, m in zip(axs, _ORD):
        mu, sd = ETA_PRIOR[m]; rng = np.random.default_rng(SEED)
        eta = rng.normal(mu, sd, 200000)
        logC = rng.normal(rng.normal(rng.normal(eta, np.abs(rng.normal(0, .6, 200000))),
                          np.abs(rng.normal(0, .6, 200000))), np.abs(rng.normal(0, 1., 200000)))
        ax.hist(logC, bins=120, range=(-6, 12), density=True, color=_MC[m], alpha=0.5, label="prior predictive")
        d = df[df.matrix == m]; det = np.log10(d[d.detection == 1].conc.astype(float))
        if len(det): ax.axvspan(det.min(), det.max(), color="#4C9A5A", alpha=0.25, label="observed")
        if lit[m]: ax.axvspan(np.log10(lit[m][0]), np.log10(lit[m][1]), color="#E07A5F", alpha=0.2, label="literature")
        ax.axvline(mu, color="k", ls="--", lw=1.3, label="prior median")
        ax.set_title(f"{m} ({UNIT[m]})", color=_MC[m], fontweight="bold", fontsize=10)
        ax.set_xlabel("log10 concentration"); ax.set_yticks([]); ax.legend(fontsize=7)
    fig.suptitle("(5) Prior predictive check — prior brackets observed & literature ranges", fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.94]); fig.savefig(fname, dpi=170); return fig


# ---- (6) prior sensitivity (needs bhm_prior_sensitivity.csv / sens_tbl) -----
def plot_prior_sensitivity(sens_tbl, fname="fig6_priorsens.png"):
    mats = [m for m in _ORD if m in sens_tbl.matrix.unique()]
    fig, ax = plt.subplots(figsize=(8.5, 4.4))
    for m in mats:
        t = sens_tbl[sens_tbl.matrix == m].sort_values("prior_shift_log10")
        ax.plot(t["prior_shift_log10"], t["post_grand_mean"], "o-", color=_MC[m], label=m)
        ax.fill_between(t["prior_shift_log10"], t["gm_lo"], t["gm_hi"], color=_MC[m], alpha=0.15)
    ax.set_yscale("log"); ax.set_xticks([-1, 0, 1])
    ax.set_xticklabels(["-1 (×0.1)", "0 (base)", "+1 (×10)"])
    ax.set_xlabel("prior median shift (log10)"); ax.set_ylabel("posterior grand mean (unit)")
    ax.set_title("(6) Prior sensitivity — flat lines across a 100× prior swing ⇒ data-dominated",
                 fontweight="bold")
    ax.legend(fontsize=9); fig.tight_layout(); fig.savefig(fname, dpi=170); return fig


# ---- (6-alt) prior sensitivity as a robustness "tornado" (compact) ----------
def plot_prior_sensitivity_tornado(sens_tbl, fname="fig6b_priorsens_tornado.png"):
    """Alternative to the line plot: horizontal bars of the posterior grand-mean
    fold-change across the full 100x prior swing (x0.1 -> x10). A short bar near
    1x = robust/data-dominated; a long bar = prior-sensitive."""
    mats = [m for m in _ORD if m in sens_tbl.matrix.unique()]
    folds = []
    for m in mats:
        s = sens_tbl[sens_tbl.matrix == m].set_index("prior_shift_log10")
        lo = s.loc[-1.0, "post_grand_mean"]; hi = s.loc[1.0, "post_grand_mean"]
        folds.append(hi / max(lo, 1e-9))
    fig, ax = plt.subplots(figsize=(8, 3.8))
    y = np.arange(len(mats))
    ax.barh(y, folds, color=[_MC[m] for m in mats], alpha=0.8)
    ax.axvline(1, color="k", lw=1)                                # no change
    ax.axvspan(1/1.5, 1.5, color="green", alpha=0.08, label="±1.5× (robust)")
    ax.axvline(10, color="grey", ls="--", lw=1, label="10× (= prior sensitive)")
    ax.set_xscale("log"); ax.set_yticks(y); ax.set_yticklabels(mats)
    ax.set_xlabel("posterior grand-mean fold-change across a 100× prior swing")
    for yi, fo in zip(y, folds):
        ax.text(fo, yi, f"  {fo:.2g}×", va="center", fontsize=9)
    ax.set_title("(6-alt) Prior robustness — fold-change of posterior vs 100× prior swing\n"
                 "bars near 1× ⇒ data-dominated / robust", fontweight="bold", fontsize=10)
    ax.legend(fontsize=8, loc="lower right"); fig.tight_layout(); fig.savefig(fname, dpi=170); return fig


def plot_all(idatas, compare_results=None, byfarm_tbl=None, sens_tbl=None,
             exposure_factors=None, show=False):
    """One call to render figures 1,2,3a,4a (and 3b,4b,6 if their inputs are given)."""
    figs = {"1_grandmean": plot_grandmean(idatas),
            "2_variance": plot_variance(idatas),
            "3a_zero_inflation": plot_zero_inflation(idatas),
            "4a_exposure": plot_exposure(idatas, exposure_factors),
            "5_prior_predictive": plot_prior_predictive()}
    if compare_results: figs["3b_loo"] = plot_loo_compare(compare_results)
    if byfarm_tbl is not None: figs["4b_by_farm"] = plot_by_farm(byfarm_tbl)
    if sens_tbl is not None:
        figs["6_prior_sensitivity"] = plot_prior_sensitivity(sens_tbl)
        figs["6b_prior_sensitivity_tornado"] = plot_prior_sensitivity_tornado(sens_tbl)
    print("Saved figures:", ", ".join(f"fig*{k.split('_')[0]}*.png" for k in figs))
    if show:
        import matplotlib.pyplot as plt; plt.show()
    return figs


if __name__ == "__main__":
    summary, idatas = run_all()          # 1  refit (all 10 farms) + grand mean
    report_variance(idatas)              # 2  tau / omega / sigma  median [95% CrI]
    report_zero_inflation(idatas)        # 3  zero-inflation posterior
    report_exposure(idatas)              # 4  occurrence / conditional / marginal (matrix)
    byfarm = report_by_farm(idatas)      # 4  per-farm
    prior_predictive_check()             # 5  prior predictive check
    sens = prior_sensitivity()           # 6  prior sensitivity (eta median +/-1 log10)
    comp = {m: compare_models(m)[0] for m in ["Milk", "Water", "Bioaerosol", "Dust"]}  # 3 LOO-CV

    # ---- ALL figures for points 1-6 (saved as PNG; inline on Colab) ----
    plot_all(idatas, compare_results=comp, byfarm_tbl=byfarm, sens_tbl=sens)



=================  Milk  =================


Output()

ERROR:pymc.stats.convergence:There were 3 divergences after tuning. Increase `target_accept` or reparameterize.


        mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
eta    1.974  0.720   0.609    3.341      0.023    0.016  1121.473   917.691  1.003
tau    0.524  0.402   0.000    1.260      0.010    0.009  2229.894  1798.300  1.002
omega  1.674  0.371   1.033    2.373      0.014    0.021  1107.735   430.329  1.003
sigma  0.533  0.127   0.331    0.770      0.002    0.002  6858.902  4704.807  1.001
delta  1.773  0.842   0.000    3.056      0.021    0.009  1555.204  1764.821  1.002
  farms kept = 10/10 (0-detection farms still in model: ['F-01', 'F-03', 'F-04', 'F-06'])
  grand-mean CFU/mL: median 84.98  95% CrI [4.85, 3845]
  diagnostics: max R-hat 1.0030 | min ESS 1108 | divergences 3

=================  Water  =================


Output()

...

/usr/local/lib/python3.13/dist-packages/arviz/stats/stats.py:797: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/arviz/stats/stats.py:797: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


(higher elpd = better; rank 0 = preferred model)
                rank   elpd_loo      p_loo  elpd_diff        weight         se       dse  warning scale
censoring_only     0 -38.479524  24.905981    0.00000  1.000000e+00  10.332556  0.000000     True   log
zero_inflated      1 -51.878943  27.334934   13.39942  3.907985e-14  10.900824  0.897485     True   log
  -> preferred by LOO-CV: censoring only  (zero inflated worse by |elpd_diff| = 13.4, dse = 0.9)
  If Pareto k > 0.7 is flagged, LOO is approximate for those influential points.

====  Model comparison Dust: zero-inflated vs censoring-only (LOO-CV)  ====


Output()

Output()

(higher elpd = better; rank 0 = preferred model)
                rank    elpd_loo      p_loo  elpd_diff   weight        se       dse  warning scale
zero_inflated      0 -123.483043  15.678293   0.000000  0.61346  5.103527  0.000000    False   log
censoring_only     1 -125.493958  12.254595   2.010915  0.38654  4.070243  5.028726    False   log
  -> preferred by LOO-CV: zero inflated  (censoring only worse by |elpd_diff| = 2.0, dse = 5.0)
     (difference < 2*dse: models are close; report as weak/moderate support)
  If Pareto k > 0.7 is flagged, LOO is approximate for those influential points.
Saved figures: fig*1*.png, fig*2*.png, fig*3a*.png, fig*4a*.png, fig*5*.png, fig*3b*.png, fig*4b*.png, fig*6*.png, fig*6b*.png
